# TradeDQN — Results Analysis

> ⚠️ **Teaching tool, not investment advice.** Past performance does not predict future results.

This notebook is the research write-up for the project (§9): it states the RL formulation with the
governing equations, reports the **held-out** backtest, runs a **one-at-a-time sensitivity sweep** on the
validation split, and analyses the **in-sample vs out-of-sample** gap. It reads the artefacts produced by
`scripts/generate_results.py` and `scripts/parameter_sweep.py` (so it runs fast); regenerate them with:

```bash
uv run python scripts/generate_results.py --episodes 120
uv run python scripts/parameter_sweep.py --episodes 15
```

## 1. The RL formulation

The agent observes a state $s_t \in \mathbb{R}^{30\times 10}$ (30 trading days × 10 features), chooses
$a \in \{\text{Sell}=0,\ \text{Hold}=1,\ \text{Buy}=2\}$, and learns $Q(s,a;\theta)$ with a **Dueling DQN**.

**Dueling aggregation** (value + advantage):
$$ Q(s,a;\theta) = V(s) + \Big( A(s,a) - \tfrac{1}{|\mathcal{A}|}\sum_{a'} A(s,a') \Big) $$

**Bellman target** with a frozen target network $\theta^{-}$ and **MSE loss**:
$$ y = r + \gamma \max_{a'} Q(s',a';\theta^{-})\,(1-\text{done}), \qquad \mathcal{L}(\theta) = \mathbb{E}\big[(y - Q(s,a;\theta))^2\big] $$

**Reward** (risk- and cost-adjusted, not raw profit):
$$ r_t = \Delta V_t - C_t - S_t + \lambda\, \text{Sharpe}_t $$
where $\Delta V_t$ is the portfolio value change, $C_t$ transaction cost, $S_t$ slippage, and
$\text{Sharpe}_t = \dfrac{\overline{r}}{\sigma_r}$ the rolling risk-adjusted return.

**Exploration** is $\varepsilon$-greedy: a random action with probability $\varepsilon$, else $\arg\max_a Q(s,a;\theta)$,
with $\varepsilon$ decayed each episode toward a floor.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# locate the project root (works whether run from notebooks/ or the repo root)
root = Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
ANALYSIS = root / 'results' / 'analysis'
metrics = json.loads((ANALYSIS / 'backtest_metrics.json').read_text())
metrics

## 2. Held-out backtest

The policy is trained on the train slice and evaluated **greedy** on the held-out **test** slice it never
saw. The honest question is not "does the line go up" but whether the DQN beats a **Buy & Hold** benchmark
on a **risk-adjusted** basis (Sharpe), trades economically, and generalises.

**Falsifiable hypotheses:**
- **H1** — the DQN's held-out total return exceeds Buy & Hold. 
- **H2** — the DQN's held-out Sharpe is positive.

In [ ]:
summary = pd.DataFrame({
    'metric': ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'num_trades'],
    'DQN':    [metrics['total_return'], metrics['sharpe_ratio'], metrics['max_drawdown'],
              metrics['win_rate'], metrics['num_trades']],
    'Buy&Hold': [metrics['benchmark_return'], None, None, None, 1],
})
display(summary)
h1 = metrics['total_return'] > metrics['benchmark_return']
h2 = metrics['sharpe_ratio'] > 0
print(f"H1 (DQN return > Buy&Hold): {'SUPPORTED' if h1 else 'REJECTED'}")
print(f"H2 (DQN Sharpe > 0):        {'SUPPORTED' if h2 else 'REJECTED'}")
display(Image(filename=str(ANALYSIS / 'backtest_equity.png')))

## 3. Sensitivity analysis (one-at-a-time, on validation)

We vary one hyperparameter at a time (`learning_rate`, `gamma`), holding the rest at config defaults, and
measure **validation** Sharpe — never the test set, so tuning cannot leak into the reported test result.

**Falsifiable hypothesis H3** — validation Sharpe is sensitive to the learning rate (it is not flat across the grid).

In [ ]:
sweep_path = ANALYSIS / 'sweep.json'
if sweep_path.exists():
    sweep = json.loads(sweep_path.read_text())
    for param, rows in sweep.items():
        print(f'\n{param}:')
        display(pd.DataFrame(rows))
    best_lr = max(sweep['learning_rate'], key=lambda r: r['val_sharpe'])
    spread = max(r['val_sharpe'] for r in sweep['learning_rate']) - min(r['val_sharpe'] for r in sweep['learning_rate'])
    print(f"\nBest val learning_rate = {best_lr['value']} (Sharpe {best_lr['val_sharpe']:+.3f})")
    print(f"H3 (val Sharpe sensitive to lr): {'SUPPORTED' if spread > 0.05 else 'TENTATIVE'} (spread {spread:.3f})")
    display(Image(filename=str(ANALYSIS / 'sweep.png')))
else:
    print('Run: uv run python scripts/parameter_sweep.py --episodes 15')

## 4. In-sample vs out-of-sample (overfitting)

On the **training** period the policy compounded the starting capital into the millions by the final
episodes (see `results/generate.log` / the README training curve), yet on the unseen **test** slice it
barely moved. That gap is the headline finding: the agent **memorised** profitable sequences specific to the
training window that do not generalise — the classic overfitting failure mode (learn the data manifold,
don't memorise isolated points). The sensitivity sweep + early-stopping on **validation** Sharpe (not
training reward) are the levers to reduce it; training across multiple tickers would reduce single-symbol
memorisation.

**Honesty:** the held-out result is reported exactly as measured — the DQN currently **underperforms** Buy &
Hold out-of-sample. The deliverable is a correct, honest DQN *system*, not a profitable trader. Past ≠ future.

## References

- Mnih et al. (2015), *Human-level control through deep reinforcement learning*, Nature — DQN (experience replay + target network).
- Wang et al. (2016), *Dueling Network Architectures for Deep RL* — the value/advantage split used here.
- Sutton & Barto (2018), *Reinforcement Learning: An Introduction*, 2nd ed.
- Watkins & Dayan (1992), *Q-learning*.
- Fischer (2018), survey on *Reinforcement Learning in Financial Markets*.
- Hugging Face *Deep RL Course*, Unit 3.